In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler

In [2]:
customer=pd.read_csv('/content/drive/MyDrive/customer_data.csv')

In [3]:
customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   customer_id        100000 non-null  int64  
 1   age                100000 non-null  int64  
 2   gender             100000 non-null  object 
 3   country            100000 non-null  object 
 4   education_level    100000 non-null  object 
 5   subscription_type  100000 non-null  object 
 6   income             100000 non-null  float64
 7   credit_score       100000 non-null  int64  
 8   purchase_amount    100000 non-null  float64
dtypes: float64(2), int64(3), object(4)
memory usage: 6.9+ MB


# Task 1: Discretization (Binning)

The `age` column contains continuous numerical values. To simplify the analysis, the ages are discretized into three logical bins:

* **Young:** 18–35
* **Adult:** 36–55
* **Senior:** 56 and above

The resulting categorical column is stored as **`age_binned`**.

In [4]:
def categorize_age(age):
    if 18 <= age <= 35:
        return "Young"
    elif 36 <= age <= 55:
        return "Adult"
    elif age >= 56:
        return "Senior"
    else:
        return "Unknown"

customer["age_binned"] = customer["age"].apply(categorize_age)


# Task 2: Label Encoding

The `gender` and `subscription_type` columns contain categorical text data. Label Encoding is used to convert each unique category into an integer value.

The encoded columns are saved as **`gender_le`** and **`subscription_le`**.


In [5]:
gender_encoder = LabelEncoder()
subscription_encoder = LabelEncoder()
customer["gender_le"] = gender_encoder.fit_transform(customer["gender"])
customer["subscription_le"] = subscription_encoder.fit_transform( customer["subscription_type"] )

In [6]:
print(dict(zip(gender_encoder.classes_, gender_encoder.transform(gender_encoder.classes_))))

{'Female': np.int64(0), 'Male': np.int64(1), 'Other': np.int64(2)}


In [7]:
print(dict(zip(subscription_encoder.classes_, subscription_encoder.transform(subscription_encoder.classes_))))

{'Basic': np.int64(0), 'Free': np.int64(1), 'Premium': np.int64(2)}


# Task 3: One-Hot Encoding

The `country` column represents **nominal categorical data**, meaning that the categories do not have any inherent order. One-Hot Encoding is used to convert each unique country into a separate binary (0/1) column.

To avoid the **dummy variable trap**, the first category is dropped using `drop_first=True`. This is particularly important when the encoded data is later used with linear regression or other linear models.

In [8]:
country_encoded = pd.get_dummies(customer["country"],prefix="country",drop_first=True)
customer = pd.concat([customer, country_encoded], axis=1)

# Task 4: Ordinal Encoding

The `education_level` column contains categorical data with an inherent order:

**High School < Bachelor < Master < PhD**

Since the categories have a meaningful ranking, Ordinal Encoding is used to convert them into integer values from **0 to 3**. The encoded values are stored in a new column called **`education_ordinal`**.

In [9]:
education_order = [["High School", "Bachelor", "Master", "PhD"]]
ordinal_encoder = OrdinalEncoder(categories=education_order)
customer["education_ordinal"] = ordinal_encoder.fit_transform(customer[["education_level"]]).astype(int)

# Task 5: Z-Score Standardization

The `credit_score` column operates on a specific numerical scale. To standardize it, **Z-Score Standardization** is applied so that the transformed values have a mean of approximately **0** and a standard deviation of **1**.

The standardized values are stored in a new column called **`credit_score_z`**.

The Z-score is calculated as:

[
z = \frac{x - \mu}{\sigma}
]

where:

* (x) = original credit score
* (\mu) = mean of the credit scores
* (\sigma) = standard deviation of the credit scores

In [10]:
scaler = StandardScaler()
customer["credit_score_z"] = scaler.fit_transform(customer[["credit_score"]])

In [11]:
print("Mean:", customer["credit_score_z"].mean())
print("Standard Deviation:", customer["credit_score_z"].std())

Mean: 6.615152869926533e-16
Standard Deviation: 1.0000050000374132


# Task 6: Min-Max Normalization

The `purchase_amount` column contains numerical values that may have a wide range. **Min-Max Normalization** scales these values to a range between **0 and 1**.

The normalized values are stored in a new column called **`purchase_amount_minmax`**.

The Min-Max formula is:

[
x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
]

In [12]:
minmax_scaler = MinMaxScaler()
customer["purchase_amount_minmax"] = minmax_scaler.fit_transform(customer[["purchase_amount"]])

In [13]:
print("Minimum:", customer["purchase_amount_minmax"].min())
print("Maximum:", customer["purchase_amount_minmax"].max())

Minimum: 0.0
Maximum: 0.9999999999999999


# Task 7: Mean Normalization

**Mean Normalization** scales the `income` column using the given formula:

[
x' = \frac{x - \text{mean}}{\text{max} - \text{min}}
]

The normalized values are saved in a new column called **`income_mean_norm`**.

In [14]:
income_mean = customer["income"].mean()
income_min = customer["income"].min()
income_max = customer["income"].max()
customer["income_mean_norm"] = ((customer["income"] - income_mean) / (income_max - income_min))

# Task 8: Log Scaling

The `income` column is heavily **right-skewed**, meaning that a small number of observations have very large income values. **Log1p Scaling** is used to compress these large values and reduce the effect of the long tail.

The `Log1p` transformation calculates:

[
x' = \log(1+x)
]

Using `log1p()` is preferable to directly using `log(x)` because it safely handles zero values.

In [15]:
customer["income_log"] = np.log1p(customer["income"])

# Task 9: Decimal Scaling

**Decimal Scaling** reduces the magnitude of numerical values by dividing them by a power of 10.

First, determine the **maximum absolute value** in the `purchase_amount` column. Then find the smallest integer (j) such that:

[
\frac{\max(|x|)}{10^j} < 1
]

Finally, divide every value by (10^j).

The transformed values are stored in **`purchase_amount_decimal`**.

In [16]:
max_abs = customer["purchase_amount"].abs().max()
j = int(np.floor(np.log10(max_abs))) + 1
customer["purchase_amount_decimal"] = (customer["purchase_amount"] / (10 ** j))